<a href="https://colab.research.google.com/github/hebamuh68/Interview-Tasks/blob/main/text_classification%20/BERT_Sentiment_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# [Sentiment Analysis with BERT: A Complete Guide](https://encord.com/blog/text-classification/)
This notebook implements a movie review classifier using the IMDb dataset and the BERT model. It includes detailed explanations for each architectural step.

## Step 1: Install and Import Libraries
We need `transformers` for the model, `datasets` for the data, and `accelerate` to handle the training loop hardware.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import load_dataset
import torch

## Step 2: Load the Dataset
**Why the `.select()`?**
We use a subset (2,000 for training, 500 for testing) so that the training finishes in minutes rather than hours. This is common during the development phase to ensure your pipeline works before committing to a full 10-hour training run.

In [17]:
dataset = load_dataset("imdb")

train_dataset = dataset['train'].shuffle(seed=42).select(range(2000))
test_dataset = dataset['test'].shuffle(seed=42).select(range(500))

## Step 3: Load the Tokenizer and Model
**Why the Tokenizer?**
Models don't read words; they read numbers. The tokenizer is the "dictionary" that maps words to specific IDs that the BERT brain understands.

**Why `num_labels=2`?**
IMDb classification is a binary task: either Positive or Negative.

In [18]:
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## Step 4: Preprocess (Tokenization & Formatting)
**Why `truncation` and `padding`?**
Neural networks require every input in a batch to be the **exact same length**. Truncation cuts long text, and Padding adds zeros to short text.

**Why `set_format("torch")`?**
By default, these datasets are stored in Apache Arrow format (standard tables). Since you are using PyTorch to train your model, this converts the numerical columns into PyTorch Tensors. Without this, your model would throw a "type error" because it wouldn't recognize the data format.

In [19]:
def preprocess_function(examples):
    return tokenizer(examples["text"], truncation=True, padding=True, max_length=128)

train_dataset = train_dataset.map(preprocess_function, batched=True)
test_dataset = test_dataset.map(preprocess_function, batched=True)

train_dataset = train_dataset.remove_columns(["text"])
test_dataset = test_dataset.remove_columns(["text"])

train_dataset.set_format("torch")
test_dataset.set_format("torch")

## Step 5: Define Training Arguments
Think of `TrainingArguments` as the **"Remote Control"** for your model's training process. Instead of hard-coding values inside the training loop, you consolidate all the "knobs and dials" here.

Here is a breakdown of what each setting does and why it matters:

---

### **1. The Logistics (Where and When)**
* **`output_dir="./results"`**: This is the folder where your model checkpoints, configurations, and results will be saved.
* **`logging_dir='./logs'`**: This saves data for visualization tools (like TensorBoard). It’s how you see those cool graphs of your model's progress.
* **`logging_steps=10`**: Tells the model to report its progress (loss, speed, etc.) every 10 steps. If you set this too high, you’ll be sitting in silence for a long time wondering if it's working!

---

### **2. The Learning Strategy (The "Brain" Settings)**
* **`learning_rate=2e-5`**: This is arguably the most important setting. It controls how much the model adjusts its weights after seeing an error.
    * **Too high:** The model overcorrects and "explodes."
    * **Too low:** The model learns so slowly it might never finish.
    * $2 \times 10^{-5}$ ($0.00002$) is a standard "sweet spot" for fine-tuning models like BERT.
* **`num_train_epochs=3`**: An **epoch** is one full pass through your entire dataset. Training for 3 epochs means the model will see every sentence in your training set 3 times.
* **`weight_decay=0.01`**: This is a technique to prevent **overfitting**. It slightly penalizes the model for having weights that are too large, forcing it to stay "simple" and generalize better to new data.

---

### **3. Efficiency and Hardware**
* **`per_device_train_batch_size=16`**: This is how many examples the model looks at simultaneously before updating its internal math.
    * **Why 16?** It fits well in the memory (VRAM) of most modern GPUs. If you get an "Out of Memory" error, you’d lower this to 8 or 4.

---

### **4. Evaluation and Saving (Quality Control)**
* **`evaluation_strategy="epoch"`**: Tells the model to run the "test" (validation set) at the end of every epoch. This helps you see if the model is actually getting smarter or just memorizing the training data.
* **`save_strategy="epoch"`**: Saves a version of the model (a checkpoint) every time an epoch finishes.
* **`load_best_model_at_end=True`**: This is your safety net. Training usually gets better and then starts to get worse if you go too long. This setting ensures that when the code finishes, you are left with the version of the model that had the **lowest error**, not just the very last one.

In [20]:
training_args = TrainingArguments(
    output_dir = "./results",
    logging_dir = "./logs",
    logging_steps = 10,

    learning_rate =2e-5,
    num_train_epochs = 1,
    weight_decay = 0.01,

    per_device_train_batch_size = 16,
    per_device_eval_batch_size = 16,

    eval_strategy = "epoch",
    save_strategy = "epoch",
    load_best_model_at_end = True
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


## Step 6: Initialize and Train
We wrap everything in the `Trainer` class which automates the training loop for us.

In [21]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
)

trainer.train()

Epoch,Training Loss,Validation Loss
1,0.387491,0.379632


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=125, training_loss=0.5164403114318847, metrics={'train_runtime': 91.5097, 'train_samples_per_second': 21.856, 'train_steps_per_second': 1.366, 'total_flos': 131555527680000.0, 'train_loss': 0.5164403114318847, 'epoch': 1.0})

## Step 7: Test Accuracy
**Why `argmax(axis=-1)`?**
The model outputs raw scores (logits) for each class. `argmax` looks across the columns (axis=-1) for each individual row to find the index of the highest score. If index 1 is higher, the prediction is Positive.

In [22]:
predictions, labels, _ = trainer.predict(test_dataset)
predicted_classes = predictions.argmax(axis=-1)
accuracy = (predicted_classes == labels).mean()
print(f"Test Accuracy: {accuracy:.4f}")

Test Accuracy: 0.8380


## Step 8: Sample Inference
**Why the device check?**
If your model is on the GPU but your input string is on the CPU, the math will fail. This code ensures both are in the same "room" (hardware device).

In [23]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

sample_text = "This movie was amazing, I loved it!"
inputs = tokenizer(sample_text, return_tensors="pt", truncation=True, padding=True, max_length=128)
inputs = {key: value.to(device) for key, value in inputs.items()}

with torch.no_grad():
    output = model(**inputs)

prediction = output.logits.argmax(dim=1).item()
print(f"Prediction: {'Positive' if prediction == 1 else 'Negative'}")

Prediction: Positive
